# Predict - BasicTS fast training on Colab Pro

This notebook is only the GPU training runner. The project code stays in the repo. Run one model/dataset pair first, inspect the log, then continue through `RUN_ORDER`.

Default first run: `STID` on `METR-LA`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, subprocess, time, json, shutil, re, csv, glob
import pandas as pd

DRIVE = pathlib.Path('/content/drive/MyDrive/dhakaflow/predict')
DRIVE.mkdir(parents=True, exist_ok=True)
RAW = DRIVE / 'raw_data'
RESULTS = DRIVE / 'results'
LOGS = DRIVE / 'logs'
CKPT = DRIVE / 'ckpt'
for p in (RAW, RESULTS, LOGS, CKPT):
    p.mkdir(parents=True, exist_ok=True)

RUN_ORDER = [
    ('STID', 'METR-LA'),
    ('STID', 'PEMS-BAY'),
    ('GWNet', 'METR-LA'),
    ('GWNet', 'PEMS-BAY'),
    ('DCRNN', 'METR-LA'),
    ('DCRNN', 'PEMS-BAY'),
    ('AGCRN', 'METR-LA'),
    ('STAEformer', 'METR-LA'),
]

TARGET_MODEL = 'STID'
TARGET_DATASET = 'METR-LA'
FORCE_RERUN = False
RUN_ALL_READY_PAIRS = False

print('Drive folder:', DRIVE)
!nvidia-smi -L


## 1. Install pinned BasicTS legacy fair-comparison harness

In [ ]:
BASICTS_SHA = '4c3e2d373c647667c2123e1591f2d79c229dfcc6'
BASICTS_REPO = 'https://github.com/GestaltCogTeam/BasicTS.git'
%cd /content
if not pathlib.Path('/content/BasicTS/.git').exists():
    !git clone -q $BASICTS_REPO /content/BasicTS
%cd /content/BasicTS
!git fetch -q --all --tags
!git checkout -q $BASICTS_SHA
!git log --oneline -1
!pip install -q -U pip wheel setuptools
!pip install -q -r requirements.txt
!pip install -q gdown
!pip install -q -e .
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'


## 2. Prepare one dataset

Put BasicTS raw data under `MyDrive/dhakaflow/predict/raw_data/`. For the first run only `raw_data/METR-LA` is required. The cell prepares only `TARGET_DATASET`, not every dataset.

In [ ]:
REQUIRED_RAW_FILES = {
    'METR-LA': ['METR-LA.h5', 'adj_METR-LA.pkl'],
    'PEMS-BAY': ['PEMS-BAY.h5', 'adj_PEMS-BAY.pkl'],
    'PEMS08': ['PEMS08.npz', 'adj_PEMS08.pkl'],
}

DCRNN_DRIVE_FOLDER_ID = '10FOTa6HXPqX8Pf5WRoRwcFnW9BrNZEIX'
H5_ALIASES = {
    'METR-LA': ['METR-LA.h5', 'metr-la.h5'],
    'PEMS-BAY': ['PEMS-BAY.h5', 'pems-bay.h5'],
}

def try_download_dcrnn_h5(dataset: str):
    if dataset not in H5_ALIASES:
        return
    ds_dir = RAW / dataset
    ds_dir.mkdir(parents=True, exist_ok=True)
    expected = ds_dir / H5_ALIASES[dataset][0]
    if expected.exists():
        return
    print('Trying official DCRNN Google Drive folder for', dataset)
    import gdown
    tmp = RAW / '_dcrnn_gdrive'
    tmp.mkdir(parents=True, exist_ok=True)
    try:
        gdown.download_folder(id=DCRNN_DRIVE_FOLDER_ID, output=str(tmp), quiet=False, use_cookies=False)
    except Exception as ex:
        print('Automatic gdown download failed:', repr(ex))
    found = None
    wanted = {name.lower() for name in H5_ALIASES[dataset]}
    for p in tmp.rglob('*.h5'):
        if p.name.lower() in wanted:
            found = p
            break
    if found:
        shutil.copy2(found, expected)
        print('Copied', found, '->', expected)

def require_dataset(dataset: str):
    try_download_dcrnn_h5(dataset)
    ds_dir = RAW / dataset
    expected = REQUIRED_RAW_FILES.get(dataset, [])
    missing = [name for name in expected if not (ds_dir / name).exists()]
    if not ds_dir.exists() or missing:
        print('Missing raw dataset files for', dataset)
        print('Folder:', ds_dir)
        if missing:
            print('Missing files:', missing)
        print('Expected layout: MyDrive/dhakaflow/predict/raw_data/' + dataset + '/...')
        print('Use BasicTS datasets/README.md links, or upload with:')
        print('  rclone copy <local-' + dataset + '-folder> gdrive2tb:dhakaflow/predict/raw_data/' + dataset)
        print('\nBasicTS dataset README preview:')
        readme = pathlib.Path('datasets/README.md')
        if readme.exists():
            print('\n'.join(readme.read_text(errors='ignore').splitlines()[:120]))
        raise FileNotFoundError(ds_dir)
    return ds_dir

def prepare_dataset(dataset: str):
    require_dataset(dataset)
    link = pathlib.Path('datasets/raw_data')
    if link.exists() or link.is_symlink():
        if link.is_symlink() or link.is_file():
            link.unlink()
        else:
            shutil.rmtree(link)
    link.symlink_to(RAW, target_is_directory=True)
    prep = pathlib.Path(f'scripts/data_preparation/{dataset}/generate_training_data.py')
    if prep.exists():
        print('Preparing', dataset, 'with', prep)
        subprocess.run(['python', str(prep)], check=True)
    else:
        print('No preparation script for', dataset, '- assuming prepared files already exist')

prepare_dataset(TARGET_DATASET)


## 3. Train one selected pair

In [ ]:
def config_path(model: str, dataset: str) -> pathlib.Path:
    candidates = [
        pathlib.Path(f'baselines/{model}/{dataset}.py'),
        pathlib.Path(f'baselines/{model.lower()}/{dataset}.py'),
        pathlib.Path(f'examples/{model}/{dataset}.py'),
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError('No BasicTS config found for ' + model + '/' + dataset + ': ' + ', '.join(map(str, candidates)))

def train_pair(model: str, dataset: str) -> dict:
    cfg = config_path(model, dataset)
    log = LOGS / f'{model}_{dataset}.log'
    manifest = RESULTS / f'{model}_{dataset}.run.json'
    if log.exists() and not FORCE_RERUN and 'return_code: 0' in log.read_text(errors='ignore')[-4000:]:
        print('Already completed:', model, dataset, log)
        return json.loads(manifest.read_text()) if manifest.exists() else {'model': model, 'dataset': dataset, 'status': 'already_completed'}
    run = {
        'model': model,
        'dataset': dataset,
        'config': str(cfg),
        'basicts_sha': BASICTS_SHA,
        'started_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'log': str(log),
    }
    cmd = ['python', 'experiments/train.py', '-c', str(cfg), '-g', '0']
    print('TRAIN:', ' '.join(cmd))
    t0 = time.time()
    with open(log, 'w') as fh:
        fh.write('command: ' + ' '.join(cmd) + '\n')
        fh.write('model: ' + model + '\ndataset: ' + dataset + '\n')
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert proc.stdout is not None
        tail = []
        for line in proc.stdout:
            fh.write(line)
            tail.append(line.rstrip())
            tail = tail[-25:]
            if any(k in line.lower() for k in ('epoch', 'mae', 'rmse', 'mape', 'test', 'best')):
                print(line.rstrip())
        rc = proc.wait()
        fh.write('\nreturn_code: ' + str(rc) + '\n')
    run.update({
        'return_code': rc,
        'status': 'PASS' if rc == 0 else 'FAIL',
        'wall_minutes': round((time.time() - t0) / 60, 2),
        'finished_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    })
    ckpt_src = pathlib.Path('checkpoints') / model
    if ckpt_src.exists():
        ckpt_dst = CKPT / model / dataset
        if ckpt_dst.exists():
            shutil.rmtree(ckpt_dst)
        shutil.copytree(ckpt_src, ckpt_dst)
        run['checkpoint_dir'] = str(ckpt_dst)
    manifest.write_text(json.dumps(run, indent=1))
    if rc != 0:
        print('Last log lines:')
        print('\n'.join(log.read_text(errors='ignore').splitlines()[-120:]))
        raise SystemExit(rc)
    print('DONE:', run)
    return run

pairs = RUN_ORDER if RUN_ALL_READY_PAIRS else [(TARGET_MODEL, TARGET_DATASET)]
for model, dataset in pairs:
    TARGET_MODEL, TARGET_DATASET = model, dataset
    prepare_dataset(dataset)
    train_pair(model, dataset)


## 4. Build `benchmark.csv` and `benchmark.md` from logs

In [ ]:
PUBLISHED_MAE = {
    ('GWNet', 'METR-LA', 15): 2.69,
    ('GWNet', 'METR-LA', 30): 3.07,
    ('GWNet', 'METR-LA', 60): 3.53,
    ('GWNet', 'PEMS-BAY', 15): 1.30,
    ('GWNet', 'PEMS-BAY', 30): 1.63,
    ('GWNet', 'PEMS-BAY', 60): 1.95,
    ('DCRNN', 'METR-LA', 15): 2.77,
    ('DCRNN', 'METR-LA', 30): 3.15,
    ('DCRNN', 'METR-LA', 60): 3.60,
    ('DCRNN', 'PEMS-BAY', 15): 1.38,
    ('DCRNN', 'PEMS-BAY', 30): 1.74,
    ('DCRNN', 'PEMS-BAY', 60): 2.07,
}

def parse_log(path: pathlib.Path):
    model, dataset = path.stem.split('_', 1)
    txt = path.read_text(errors='ignore')
    rows = []
    patterns = [
        r'horizon\s*(\d+).*?MAE\s*[:=]\s*([0-9.]+).*?RMSE\s*[:=]\s*([0-9.]+).*?MAPE\s*[:=]\s*([0-9.]+)',
        r'Horizon\s*(\d+).*?mae\s*[:=]\s*([0-9.]+).*?rmse\s*[:=]\s*([0-9.]+).*?mape\s*[:=]\s*([0-9.]+)',
    ]
    seen = set()
    for pat in patterns:
        for h, mae, rmse, mape in re.findall(pat, txt, re.S | re.I):
            h = int(h)
            horizon_min = h * 5 if h in (3, 6, 12) else h
            if horizon_min not in (15, 30, 60):
                continue
            key = (model, dataset, horizon_min)
            if key in seen:
                continue
            seen.add(key)
            rows.append({
                'model': model,
                'dataset': dataset,
                'horizon_min': horizon_min,
                'mae': float(mae),
                'rmse': float(rmse),
                'mape': float(mape),
                'published_mae': PUBLISHED_MAE.get((model, dataset, horizon_min), ''),
                'log': str(path),
            })
    return rows

rows = []
for log in sorted(LOGS.glob('*.log')):
    rows.extend(parse_log(log))
rows = sorted(rows, key=lambda r: (r['model'], r['dataset'], r['horizon_min']))
fields = ['model', 'dataset', 'horizon_min', 'mae', 'rmse', 'mape', 'published_mae', 'log']
with open(RESULTS / 'benchmark.csv', 'w', newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=fields)
    w.writeheader()
    w.writerows(rows)
if rows:
    df = pd.DataFrame(rows)
    (RESULTS / 'benchmark.md').write_text(df.to_markdown(index=False) + '\n')
    display(df)
else:
    (RESULTS / 'benchmark.md').write_text('_No parsed benchmark rows yet. Inspect logs in Drive._\n')
    print('No parsed benchmark rows yet. Latest logs:', sorted(map(str, LOGS.glob('*.log')))[-3:])
print('wrote', RESULTS / 'benchmark.csv')
print('wrote', RESULTS / 'benchmark.md')


## 5. Copy results back to the repo

On the Mac, after this notebook writes Drive results, run:

```bash
python predict/pull_colab_results.py --drive /path/to/MyDrive/dhakaflow/predict
# or use rclone copy gdrive2tb:dhakaflow/predict/results predict/results
```
